# Ajuste LoRA de Stable Diffusion con Pokémon

El cuaderno ejecuta el script oficial de Diffusers para ajustar una adaptación LoRA sobre un subconjunto del dataset Pokémon con descripciones BLIP. La base de Stable Diffusion v1.5 es un modelo con acceso condicionado: antes de correr, acepta sus condiciones en Hugging Face y configura el secreto HF_TOKEN en Colab. No pegues el token en una celda ni lo publiques. Una corrida breve permite probar el flujo; no representa un ajuste exhaustivo.

In [ ]:
!pip -q install -U diffusers transformers datasets accelerate peft safetensors huggingface_hub
!git clone -q --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers
!pip -q install -e /content/diffusers

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import torch, os, json, glob, zipfile

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Agrega HF_TOKEN en Secretos de Colab y habilita su acceso a este cuaderno.')
login(token=HF_TOKEN, add_to_git_credential=False)
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
MODEL='runwayml/stable-diffusion-v1-5'
DATASET='lambdalabs/pokemon-blip-captions'
OUT='/content/lora-pokemon'

## Entrenamiento

El script conserva congelados los pesos base y entrena matrices de bajo rango en las capas de atención. Se limita el experimento a 200 imágenes y 200 pasos para que sea viable en una GPU T4; anota las limitaciones al interpretar las imágenes.

In [ ]:
%cd /content/diffusers/examples/text_to_image
!accelerate launch --mixed_precision=fp16 train_text_to_image_lora.py \
  --pretrained_model_name_or_path="$MODEL" \
  --dataset_name="$DATASET" --image_column="image" --caption_column="text" \
  --resolution=512 --center_crop --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_samples=200 --max_train_steps=200 \
  --learning_rate=1e-4 --lr_scheduler="constant" --lr_warmup_steps=0 \
  --rank=8 --seed=42 --output_dir="$OUT" \
  --validation_prompt="a small cartoon electric creature, digital illustration" \
  --validation_epochs=50 --num_validation_images=2

## Generación y descarga de evidencias

Las imágenes de abajo solo aparecerán tras completar el entrenamiento. Guarda la semilla y los prompts junto con los resultados para que la comparación sea interpretable.

In [ ]:
from diffusers import StableDiffusionPipeline
from IPython.display import display
from PIL import Image

os.makedirs('/content/resultados_lora',exist_ok=True)
pipe=StableDiffusionPipeline.from_pretrained(MODEL,torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(OUT)
prompts=['a small cartoon electric creature, digital illustration','a cute water creature in a hand-painted fantasy style']
images=[]
for i,prompt in enumerate(prompts):
    image=pipe(prompt,num_inference_steps=25,guidance_scale=7.0,generator=torch.Generator('cuda').manual_seed(120+i)).images[0]
    image.save(f'/content/resultados_lora/muestra_{i+1}.png'); images.append(image); display(image)
with open('/content/resultados_lora/registro.json','w',encoding='utf-8') as f:
    json.dump({'modelo':MODEL,'dataset':DATASET,'pasos_entrenamiento':200,'semillas':[120,121],'prompts':prompts},f,ensure_ascii=False,indent=2)
with zipfile.ZipFile('/content/resultados_lora/evidencias_lora.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob('/content/resultados_lora/*'): z.write(path,arcname=os.path.basename(path))
print('Paquete:', '/content/resultados_lora/evidencias_lora.zip')

## Lectura crítica

Compara las imágenes generadas con el contenido de los prompts: estilo, consistencia, detalles y errores visibles. Dos muestras no bastan para medir calidad general. Reporta el número de pasos, las limitaciones de memoria y cualquier error de entrenamiento. No declares el ajuste terminado si la celda de entrenamiento no finalizó.

## Referencias técnicas

- Hugging Face Diffusers, ejemplo oficial de entrenamiento LoRA: https://github.com/huggingface/diffusers/tree/main/examples/text_to_image
- Dataset Pokémon BLIP captions: https://huggingface.co/datasets/lambdalabs/pokemon-blip-captions
- Modelo Stable Diffusion v1.5: https://huggingface.co/runwayml/stable-diffusion-v1-5